# SimpleNN Hyperparameter Tuning

This notebook implements hyperparameter search for the SimpleNN quantum error correction model.

**Purpose:** Compare simple feedforward NN (no graph structure) against GraphSAGE to determine if graph structure helps.

**Workflow:**
1. Generate flat dataset cache (if not exists)
2. Run the training loop (trains all 50 configs sequentially)
3. Run the analysis cells to find the best configuration

**Distributed Mode (optional):**
Set `DISTRIBUTED_MODE = True` and `WORKER_ID` to distribute across multiple machines.

In [1]:
#==============================================================================
# MODE CONFIGURATION
#==============================================================================
DISTRIBUTED_MODE = False  # Set to True for multi-machine distributed training
WORKER_ID = 1  # Only used if DISTRIBUTED_MODE = True (set to 1-5)
#==============================================================================

if DISTRIBUTED_MODE:
    assert WORKER_ID in [1, 2, 3, 4, 5], f"WORKER_ID must be 1, 2, 3, 4, or 5, got {WORKER_ID}"
    if WORKER_ID == 5:
        my_config_ids = list(range(40, 50))
    else:
        my_config_ids = [i for i in range(40) if i % 4 == (WORKER_ID - 1)]
    print(f"DISTRIBUTED MODE - Worker {WORKER_ID}")
    print(f"This worker will train {len(my_config_ids)} configs: {my_config_ids}")
else:
    my_config_ids = list(range(50))  # Train all 50 configs
    print(f"LOCAL MODE - Training all 50 configs sequentially")

LOCAL MODE - Training all 50 configs sequentially


#### Imports

In [ ]:
# Install required libraries (uncomment and run if needed)
# !pip install stim pymatching numpy matplotlib torch tqdm

In [3]:
import sys
import json
import random
import time
from pathlib import Path
from datetime import datetime

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('..')

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

# Import from benchmark_models.py
from benchmark_models import (
    SurfaceCodeSampler,
    SimpleNNModel,
    SimpleNN,
    FlatDatasetCache,
)


# Set up paths
TUNING_DIR = BASE_PATH / "nn" / "tuning"
TUNING_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = TUNING_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = TUNING_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = TUNING_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  TUNING_DIR: {TUNING_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  MODELS_DIR: {MODELS_DIR}")

Using device: cuda
GPU: NVIDIA GeForce GTX 1080

Paths:
  BASE_PATH: ..
  TUNING_DIR: ..\nn\tuning
  RESULTS_DIR: ..\nn\tuning\results
  MODELS_DIR: ..\nn\tuning\models


#### Generate Flat Dataset Cache

This generates the flat syndrome array dataset for SimpleNN training.
Uses the same configuration as the graph datasets (d=7, 1M samples) for fair comparison.

In [ ]:
# ============================================================
# GENERATE FLAT DATASET CACHE (RUN ONCE)
# ============================================================

# Standard error rates (same as graph datasets)
STANDARD_P_VALUES = [0.001, 0.003, 0.005, 0.007]
STANDARD_P_WEIGHTS = [0.25, 0.25, 0.25, 0.25]

# Dataset config - match the GraphSAGE tuning setup
DISTANCE = 7
N_SAMPLES = 1_000_000
CACHE_NAME = f"d{DISTANCE}_baseline"

# Check if already exists
flat_datasets_dir = BASE_PATH / "datasets" / "flat"
if (flat_datasets_dir / f"{CACHE_NAME}.pt").exists():
    print(f"Flat dataset '{CACHE_NAME}' already exists!")
    print(f"Location: {flat_datasets_dir / f'{CACHE_NAME}.pt'}")
else:
    print(f"Generating flat dataset: {CACHE_NAME}")
    print(f"This may take a while...")

    cache = FlatDatasetCache(base_path=BASE_PATH, device=device)
    cache.generate(
        d=DISTANCE,
        n_samples=N_SAMPLES,
        p_values=STANDARD_P_VALUES,
        p_weights=STANDARD_P_WEIGHTS,
        verbose=True
    )
    cache.save(CACHE_NAME)
    print(f"\nFlat dataset saved!")

#### Generate Configs (RUN ONCE)

Config generation - 50 configurations for hyperparameter search.
The `configs.json` file will be created on first run.

In [4]:
# # =============================================================================
# # CONFIG GENERATION - RUN ONCE ON ANY INSTANCE
# # =============================================================================

# CONFIGS_PATH = TUNING_DIR / "configs.json"

# # Check if configs already exist
# if CONFIGS_PATH.exists():
#     print(f"configs.json already exists at {CONFIGS_PATH}")
#     print("Skipping generation. Delete the file to regenerate.")
# else:
#     # Search space definition for SimpleNN
#     # Note: SimpleNN has fixed architecture (256->512->1024->1)
#     # We vary: learning rate, batch size, and add dropout variants
#     SEARCH_SPACE = {
#         'hidden_dims': [(256, 512, 1024), (128, 256, 512), (512, 1024, 2048), (64, 128, 256)],
#         'learning_rate': [1e-4, 3e-4, 5e-4, 1e-3, 3e-3],
#         'batch_size': [64, 128, 256, 512],
#         'dropout': [0.0, 0.1, 0.2, 0.3],
#     }

#     # Fixed parameters (not searched)
#     FIXED_PARAMS = {
#         'epochs': 50,
#         'distance': 7,
#     }

#     # Number of configurations to generate
#     N_CONFIGS = 50
#     SEED = 42

#     # Set seed for reproducibility
#     random.seed(SEED)

#     # Generate random configurations
#     configs = []
#     for i in range(N_CONFIGS):
#         config = {
#             'id': i,
#             'hidden_dims': random.choice(SEARCH_SPACE['hidden_dims']),
#             'learning_rate': random.choice(SEARCH_SPACE['learning_rate']),
#             'batch_size': random.choice(SEARCH_SPACE['batch_size']),
#             'dropout': random.choice(SEARCH_SPACE['dropout']),
#         }
#         configs.append(config)

#     # Create the full config document
#     config_doc = {
#         'seed': SEED,
#         'n_configs': N_CONFIGS,
#         'generated_at': datetime.now().isoformat(),
#         'search_space': {k: [str(v) if isinstance(v, tuple) else v for v in vals] for k, vals in SEARCH_SPACE.items()},
#         'fixed_params': FIXED_PARAMS,
#         'configs': configs
#     }

#     # Save to file
#     with open(CONFIGS_PATH, 'w') as f:
#         json.dump(config_doc, f, indent=2)

#     print(f"Generated {N_CONFIGS} configurations with seed {SEED}")
#     print(f"Saved to: {CONFIGS_PATH}")
#     print(f"\nSearch space:")
#     for key, values in SEARCH_SPACE.items():
#         print(f"  {key}: {values}")
#     print(f"\nFixed parameters:")
#     for key, value in FIXED_PARAMS.items():
#         print(f"  {key}: {value}")
#     print(f"\nFirst 5 configs:")
#     for c in configs[:5]:
#         print(f"  {c}")

Generated 50 configurations with seed 42
Saved to: ..\nn\tuning\configs.json

Search space:
  hidden_dims: [(256, 512, 1024), (128, 256, 512), (512, 1024, 2048), (64, 128, 256)]
  learning_rate: [0.0001, 0.0003, 0.0005, 0.001, 0.003]
  batch_size: [64, 128, 256, 512]
  dropout: [0.0, 0.1, 0.2, 0.3]

Fixed parameters:
  epochs: 50
  distance: 7

First 5 configs:
  {'id': 0, 'hidden_dims': (256, 512, 1024), 'learning_rate': 0.0001, 'batch_size': 256, 'dropout': 0.1}
  {'id': 1, 'hidden_dims': (128, 256, 512), 'learning_rate': 0.0003, 'batch_size': 64, 'dropout': 0.0}
  {'id': 2, 'hidden_dims': (64, 128, 256), 'learning_rate': 0.0001, 'batch_size': 64, 'dropout': 0.0}
  {'id': 3, 'hidden_dims': (128, 256, 512), 'learning_rate': 0.0003, 'batch_size': 64, 'dropout': 0.1}
  {'id': 4, 'hidden_dims': (64, 128, 256), 'learning_rate': 0.0003, 'batch_size': 512, 'dropout': 0.2}


#### Training Loop

This is the main training cell. It will:
1. Load the configs assigned to this worker
2. Skip any configs already completed (for resume support)
3. Train each config and save results incrementally

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_configs():
    """Load all configurations from configs.json."""
    configs_path = TUNING_DIR / "configs.json"
    if not configs_path.exists():
        raise FileNotFoundError(f"configs.json not found at {configs_path}. Run the config generation cell first.")
    with open(configs_path, 'r') as f:
        return json.load(f)

def get_my_configs(all_configs, config_ids):
    """Get configs by ID list."""
    return [c for c in all_configs['configs'] if c['id'] in config_ids]

def get_completed_ids():
    """Get set of already-completed config IDs from results file."""
    results_path = RESULTS_DIR / "results.json"
    if not results_path.exists():
        return set()
    with open(results_path, 'r') as f:
        results = json.load(f)
    return {r['config_id'] for r in results if r.get('status') == 'completed'}

def save_result(result):
    """Append a result to the results file."""
    results_path = RESULTS_DIR / "results.json"

    if results_path.exists():
        with open(results_path, 'r') as f:
            results = json.load(f)
    else:
        results = []

    results.append(result)

    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    return len(results)

def evaluate_model(model, detections, labels, device, batch_size=256):
    """Evaluate model accuracy on a set of samples."""
    model.model.eval()

    correct = 0
    total = len(labels)

    with torch.no_grad():
        for i in range(0, total, batch_size):
            X = detections[i:i+batch_size]
            y = labels[i:i+batch_size]
            pred = model.model(X)
            correct += ((pred.squeeze() > 0.5).float() == y).sum().item()

    return correct / total if total > 0 else 0.0

def load_and_split_dataset(cache_name, train_ratio=0.8, val_ratio=0.1, seed=42):
    """Load flat dataset from cache and split into train/val/test."""
    print(f"Loading flat dataset: {cache_name}")
    cache = FlatDatasetCache(base_path=BASE_PATH, device=device)
    cache.load(cache_name, verbose=True)

    detections, labels = cache.get_data()
    n = len(labels)

    # Shuffle with fixed seed
    torch.manual_seed(seed)
    perm = torch.randperm(n, device=device)
    detections = detections[perm]
    labels = labels[perm]

    # Calculate split points
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_det = detections[:train_end]
    train_lab = labels[:train_end]
    val_det = detections[train_end:val_end]
    val_lab = labels[train_end:val_end]
    test_det = detections[val_end:]
    test_lab = labels[val_end:]

    print(f"Dataset split: {len(train_lab)} train, {len(val_lab)} val, {len(test_lab)} test")

    return (train_det, train_lab), (val_det, val_lab), (test_det, test_lab)

In [ ]:
# =============================================================================
# MODEL IMPORT
# =============================================================================
# SimpleNNModel now supports configurable hidden_dims and dropout
# Parameters: SimpleNNModel(in_channels, hidden_dims=(256, 512, 1024), dropout=0.0)
# This is imported from benchmark_models.py

In [ ]:
# =============================================================================
# LOAD CONFIGS AND DATASET
# =============================================================================

import gc

# Load all configurations
all_configs = load_configs()
fixed_params = all_configs['fixed_params']

# Get configs to train (based on mode setting from above)
my_configs = get_my_configs(all_configs, my_config_ids)
print(f"\nConfigs to train: {len(my_configs)}")
print(f"Config IDs: {[c['id'] for c in my_configs]}")

# Check which are already completed
completed_ids = get_completed_ids()
remaining_configs = [c for c in my_configs if c['id'] not in completed_ids]
print(f"\nAlready completed: {len(completed_ids)} configs")
print(f"Remaining: {len(remaining_configs)} configs")

if len(remaining_configs) == 0:
    print("\nAll configs are already completed!")
else:
    # Load dataset once (shared across all configs)
    cache_name = f"d{fixed_params['distance']}_baseline"
    (train_det, train_lab), (val_det, val_lab), (test_det, test_lab) = load_and_split_dataset(cache_name)
    num_detectors = train_det.shape[1]

In [ ]:
if len(remaining_configs) > 0:
    print(f"\n{'='*60}")
    print(f"Starting training - {len(remaining_configs)} configs")
    print(f"{'='*60}")

    for i, config in enumerate(remaining_configs):
        config_id = config['id']
        print(f"\n{'='*60}")
        print(f"Config {config_id} ({i+1}/{len(remaining_configs)})")
        print(f"{'='*60}")
        print(f"Parameters:")
        print(f"  hidden_dims: {config['hidden_dims']}")
        print(f"  learning_rate: {config['learning_rate']}")
        print(f"  batch_size: {config['batch_size']}")
        print(f"  dropout: {config['dropout']}")

        start_time = time.time()

        try:
            # Initialize model with config hyperparameters
            hidden_dims = tuple(config['hidden_dims']) if isinstance(config['hidden_dims'], list) else config['hidden_dims']

            model = SimpleNN(
                nickname=f"config_{config_id}",
                in_channels=num_detectors,
                hidden_dims=hidden_dims,
                dropout=config['dropout'],
                device=device,
                base_path=BASE_PATH,
                seed=config_id  # Use config_id as seed for reproducibility
            )

            # Train the model
            epoch_losses = model.train_from_data(
                detections=train_det,
                labels=train_lab,
                epochs=fixed_params['epochs'],
                batch_size=config['batch_size'],
                lr=config['learning_rate'],
                verbose=True
            )

            # Evaluate on validation and test sets
            val_accuracy = evaluate_model(model, val_det, val_lab, device)
            test_accuracy = evaluate_model(model, test_det, test_lab, device)

            training_time = time.time() - start_time

            # Save the model to tuning/models/
            model_path = MODELS_DIR / f"config_{config_id}.pt"
            save_dict = {
                'state_dict': model.model.state_dict(),
                'config': model._config,
                'hyperparams': config,
                'val_accuracy': val_accuracy,
                'test_accuracy': test_accuracy,
                'training_time_sec': training_time,
                'timestamp': datetime.now().isoformat()
            }
            torch.save(save_dict, model_path)
            print(f"\nModel saved to: {model_path}")

            # Create result record
            result = {
                'config_id': config_id,
                'config': config,
                'val_accuracy': val_accuracy,
                'test_accuracy': test_accuracy,
                'training_time_sec': training_time,
                'final_loss': epoch_losses[-1] if epoch_losses else None,
                'model_path': str(model_path),
                'status': 'completed',
                'timestamp': datetime.now().isoformat()
            }

            print(f"\nResults:")
            print(f"  Val Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
            print(f"  Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
            print(f"  Training Time: {training_time:.1f}s ({training_time/60:.1f} min)")

        except Exception as e:
            training_time = time.time() - start_time
            print(f"\nERROR: Config {config_id} failed: {str(e)}")
            import traceback
            traceback.print_exc()
            result = {
                'config_id': config_id,
                'config': config,
                'val_accuracy': None,
                'test_accuracy': None,
                'training_time_sec': training_time,
                'final_loss': None,
                'model_path': None,
                'status': 'failed',
                'error': str(e),
                'timestamp': datetime.now().isoformat()
            }
            # Cleanup on error too
            if 'model' in dir():
                del model

        # Save result incrementally
        n_saved = save_result(result)
        print(f"Result saved ({n_saved} total)")

        # Aggressive cleanup to prevent RAM/VRAM from filling up
        if 'model' in dir():
            del model
        if 'epoch_losses' in dir():
            del epoch_losses
        if 'save_dict' in dir():
            del save_dict

        # Force garbage collection
        gc.collect()

        # Clear CUDA cache if available
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        print(f"Memory cleared.")

    print(f"\n{'='*60}")
    print(f"TUNING COMPLETE!")
    print(f"{'='*60}")
    print(f"Total configs trained: {len(remaining_configs)}")
    print(f"Results saved to: {RESULTS_DIR / 'results.json'}")
    print(f"Models saved to: {MODELS_DIR}")

#### Analyze Results

This cell loads results (from local `results.json` or distributed worker files) and identifies the best configurations.

In [ ]:
# =============================================================================
# LOAD ALL RESULTS
# =============================================================================

def load_all_results():
    """Load results from local mode or distributed worker files."""
    all_results = []

    # First check for local results.json (local mode)
    local_results_path = RESULTS_DIR / "results.json"
    if local_results_path.exists():
        with open(local_results_path, 'r') as f:
            local_results = json.load(f)
        all_results.extend(local_results)
        print(f"Local results: {len(local_results)} configs")

    # Also check for distributed worker files
    for worker_id in [1, 2, 3, 4, 5]:
        results_path = RESULTS_DIR / f"worker_{worker_id}.json"
        if results_path.exists():
            with open(results_path, 'r') as f:
                worker_results = json.load(f)
            all_results.extend(worker_results)
            print(f"Worker {worker_id}: {len(worker_results)} results")

    # Deduplicate by config_id (in case of overlap)
    seen_ids = set()
    unique_results = []
    for r in all_results:
        if r['config_id'] not in seen_ids:
            seen_ids.add(r['config_id'])
            unique_results.append(r)

    if len(unique_results) < len(all_results):
        print(f"\n(Removed {len(all_results) - len(unique_results)} duplicate entries)")

    return unique_results

# Load results
print("Loading results...")
all_results = load_all_results()

# Filter to completed only
completed_results = [r for r in all_results if r.get('status') == 'completed']
failed_results = [r for r in all_results if r.get('status') == 'failed']

print(f"\nTotal results: {len(all_results)}")
print(f"  Completed: {len(completed_results)}")
print(f"  Failed: {len(failed_results)}")

if len(completed_results) > 0:
    # Sort by test accuracy
    sorted_results = sorted(completed_results, key=lambda x: x['test_accuracy'], reverse=True)

    print(f"\n{'='*60}")
    print(f"TOP 10 CONFIGURATIONS")
    print(f"{'='*60}")

    for i, r in enumerate(sorted_results[:10]):
        print(f"\n{i+1}. Config {r['config_id']}")
        print(f"   Test Accuracy: {r['test_accuracy']:.4f} ({r['test_accuracy']*100:.2f}%)")
        print(f"   Val Accuracy: {r['val_accuracy']:.4f}")
        print(f"   Hidden dims: {r['config']['hidden_dims']}")
        print(f"   LR: {r['config']['learning_rate']}, Batch: {r['config']['batch_size']}, Dropout: {r['config']['dropout']}")
        print(f"   Training time: {r['training_time_sec']/60:.1f} min")

In [ ]:
# =============================================================================
# SAVE COMBINED RESULTS TO CSV
# =============================================================================

if len(completed_results) > 0:
    # Create DataFrame with flattened config
    rows = []
    for r in completed_results:
        row = {
            'config_id': r['config_id'],
            'test_accuracy': r['test_accuracy'],
            'val_accuracy': r['val_accuracy'],
            'final_loss': r['final_loss'],
            'training_time_sec': r['training_time_sec'],
            'hidden_dims': str(r['config']['hidden_dims']),
            'learning_rate': r['config']['learning_rate'],
            'batch_size': r['config']['batch_size'],
            'dropout': r['config']['dropout'],
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    df = df.sort_values('test_accuracy', ascending=False)

    # Save to CSV
    csv_path = TUNING_DIR / "results_combined.csv"
    df.to_csv(csv_path, index=False)
    print(f"Results saved to: {csv_path}")

    # Display summary stats
    print(f"\nSummary Statistics:")
    print(f"  Best test accuracy: {df['test_accuracy'].max():.4f}")
    print(f"  Worst test accuracy: {df['test_accuracy'].min():.4f}")
    print(f"  Mean test accuracy: {df['test_accuracy'].mean():.4f}")
    print(f"  Std test accuracy: {df['test_accuracy'].std():.4f}")